# Step 4 of the PW: the future score of the firms

**Status: skeleton v0.1 (24 November 2026).** Imma set up the structure, the data, the outcome, the backtest and the first signals. Leo and Thomas complete it: every place to work on is marked `TODO(...)` in the code and in the text. The plan, the literature and the open decisions are in the doc *Step 4 · Future score: set-up, literature and plan*.

**The question.** Recent filings have no grants and no citations yet. Which firms of the humanoid sector hold the patents that will matter most? We score the families published in the last years with signals that are known **at filing or publication**, and we **prove that the score works** by freezing time in the past and checking what happened afterwards (the professor: "even better: prove it works").

**The design** (Lai, Wei and Hwang 2026, *Research Policy*, read in full; changes explained in the doc):
1. **Outcome** (Section 3): a family is *valuable* if it is in the top 10% of its priority year and CPC subclass by the citations it received within 5 years.
2. **Signals** (Section 4): one function each, marked *static* (fixed at publication) or *dynamic* (grows with age), each with its paper.
3. **Backtest** (Section 5): out of time. To predict the families of year T we train only on families whose 5-year outcome was already known in T. A *benchmark* (classic indicators) against a *proposed* model (classic + our signals); two models, as in Lai et al.; results with bootstrap intervals, on all of robotics and on the sector alone.
4. **Firm scores** (Section 7): the final model scores the sector's families of 2023–2025; a firm's future score sums up its families. Step 5 reads `future_score.csv`.

**Two populations.** The model learns on **the wide net** (Thomas's robotics search, 239,711 families, with the 2,907 families of the sector inside: notebook `wide_net/wide_net_v3.ipynb`), because the sector alone has about 50 highly cited families in the training years, too few to learn from. The **sector** (rule v3, Step 0) is where we give the scores and where we check that the model still works.

**Rules of the group.** PATSTAT data stay on TIP (`outputs/` and `data/` never enter git). No API key in any file. Every threshold is fixed **before** looking at the results, and every attempt stays in the notebook, kept or not.

## 1. Setup and data

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sqlalchemy import distinct, func, literal_column
from sqlalchemy.orm import aliased
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

In [ ]:
# ---------------------------------------------------------------- environment and folders
ENV = "PROD"                                  # TEST runs the code, but the numbers mean nothing

# the tables of the sector saved by Step 0 (the same search as Steps 1-3)
DATA_DIR = next((p for p in (Path("data") / ENV / "humanoid_robots_v2",
                             Path("..") / "tutorial_0" / "data" / ENV / "humanoid_robots_v2")
                 if (p / "dataset_metadata.json").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("the tables of Step 0 (data/PROD/humanoid_robots_v2) are not found")

WIDE_NET_FILE = Path("wide_net/outputs/wide_net/wide_net_families.csv")    # notebook wide_net_v3
SECTOR_IDS_FILE = Path("ids/sector_family_ids_v3_PROD.csv")               # Step 0, Section 7.4
STEP3_DIR = Path("outputs/step3")                                         # present score, firms waiting
TEXT_FILE = Path("outputs/t2_text/text_novelty_PROD_rule_v3.csv")        # notebook 21, Section 10
OUT_DIR = Path("outputs/step4")
CACHE_DIR = OUT_DIR / "cache"                 # PATSTAT answers, saved once: the second run reads them
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------- time: read from Step 0, so that all steps agree
meta = json.loads((DATA_DIR / "dataset_metadata.json").read_text())
YEAR_MIN, YEAR_MAX = meta["window"]           # 2000-2025, earliest priority year of the family
LAST_COMPLETE_YEAR = meta["last_complete_year"]   # 2022: later years are still being published

# ---------------------------------------------------------------- the outcome (Section 3)
HORIZON = 5                                   # years of forward citations, as the Influence of Step 3
LAST_LABEL_YEAR = LAST_COMPLETE_YEAR - HORIZON    # 2017: the last priority year with a complete outcome
TOP_SHARE = 0.10                              # "valuable" = top 10% of its cohort (Lai et al.: the tail)
MIN_COHORT = 30                               # a year x CPC subclass cohort smaller than this uses the year only

# ---------------------------------------------------------------- the backtest (Section 5)
TEST_YEARS = list(range(2013, LAST_LABEL_YEAR + 1))   # we predict the families of these years...
LOOKBACK = 5                                  # ...training on the LOOKBACK years whose outcome was known then
N_BOOT = 200                                  # bootstrap resamples for the intervals
RANDOM_STATE = 34                             # our group number: every random choice is reproducible

# ---------------------------------------------------------------- the firm scores (Section 7)
APPLY_YEARS = (LAST_COMPLETE_YEAR + 1, YEAR_MAX)   # 2023-2025: the families we score
MIN_RECENT_FAMILIES = 5                       # a firm needs this many recent families to get a score

# ---------------------------------------------------------------- PATSTAT
CHUNK = 5_000                                 # family IDs per query (BigQuery limits the size of a query)
DUMMY_ID_FROM = 900_000_000                   # appln_id from this value on are artificial records (as in Step 0)
DEV_SAMPLE = None                             # e.g. 20_000 to develop on a random sample of the net; None = all

print(f"env {ENV} | window {YEAR_MIN}-{YEAR_MAX} | outcome known up to {LAST_LABEL_YEAR} | "
      f"test years {TEST_YEARS[0]}-{TEST_YEARS[-1]} | scored years {APPLY_YEARS[0]}-{APPLY_YEARS[1]}")

In [ ]:
def load_table(name):
    "A table saved by Step 0: parquet, or pickle if pyarrow was missing there."
    for suffix, reader in ((".parquet", pd.read_parquet), (".pkl", pd.read_pickle)):
        path = DATA_DIR / f"{name}{suffix}"
        if path.exists():
            return reader(path)
    raise FileNotFoundError(f"{name} not found in {DATA_DIR}")


def check(name, ok):
    "Print a check in the same form as the other notebooks of the group."
    print("OK    -" if ok else "CHECK -", name)


# 1. the wide net: one row per family, with in_sector_v3 = True for the 2,907 families of the sector
net = pd.read_csv(WIDE_NET_FILE)
sector_ids = set(pd.read_csv(SECTOR_IDS_FILE)["docdb_family_id"].astype(int))
if DEV_SAMPLE:                                # develop on a sample, but always keep the whole sector
    rest = net[~net["in_sector_v3"]].sample(DEV_SAMPLE, random_state=RANDOM_STATE)
    net = pd.concat([net[net["in_sector_v3"]], rest], ignore_index=True)
net = net.set_index("docdb_family_id").sort_index()
FAMILIES = [int(f) for f in net.index]

# 2. what Steps 0, 3 and notebook 21 already computed for the sector
sector_families = load_table("families")                      # one row per family of the sector
family_applicants = load_table("family_applicants_grouped")   # one row per family and applicant (brand key)
subfields = load_table("family_subfields")                    # platform / body / control (Step 0, 7.4)
present = pd.read_csv(STEP3_DIR / "present_score.csv")        # the y axis of the quadrant
waiting = pd.read_csv(STEP3_DIR / "firms_for_step4.csv")      # the firms Step 3 could not score yet
text = pd.read_csv(TEXT_FILE) if TEXT_FILE.exists() else None

print(f"wide net: {len(net):,} families, {int(net['in_sector_v3'].sum()):,} in the sector")
print(f"firms with a present score: {len(present)} | firms waiting for Step 4: {len(waiting)}")
check("every family of the sector is in the net", sector_ids <= set(net.index))
check("the sector list and the Step 0 tables agree", sector_ids == set(sector_families["docdb_family_id"]))
check("text novelty of notebook 21 found", text is not None)

### 1.1 The PATSTAT queries

All the queries of the notebook are here, one function each, and none runs yet. Each one takes a list of family IDs, runs in chunks of 5,000 and returns a small table **grouped inside PATSTAT** (the professor: "keep heavy operations in the data platform"). `cached()` saves each answer in `outputs/step4/cache/`, so the long queries run once.

Only **patents of invention** (`ipr_type = 'PI'`) and real records (`appln_id < 900,000,000`) count, as in Steps 0–3. Citations are taken **between families** (`TLS228_DOCDB_FAM_CITN`), the same unit as the Influence of Step 3.

In [ ]:
from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (TLS201_APPLN, TLS207_PERS_APPLN, TLS209_APPLN_IPC, TLS211_PAT_PUBLN,
                                                 TLS212_CITATION, TLS215_CITN_CATEG, TLS224_APPLN_CPC,
                                                 TLS228_DOCDB_FAM_CITN)

patstat = PatstatClient(env=ENV)
db = patstat.orm()
REAL = (TLS201_APPLN.ipr_type == "PI", TLS201_APPLN.appln_id < DUMMY_ID_FROM)
SUBCLASS = func.substr(func.replace(TLS224_APPLN_CPC.cpc_class_symbol, " ", ""), 1, 4)
IPC_SUBCLASS = func.substr(func.replace(TLS209_APPLN_IPC.ipc_class_symbol, " ", ""), 1, 4)
BY_SUBCLASS = literal_column("subclass")    # group by the name of the column: BigQuery cannot group by the formula


def in_chunks(ids, make_query):
    "Run make_query(list of IDs) on chunks of IDs and stack the answers."
    ids = [int(i) for i in pd.unique(pd.Series(list(ids)))]    # plain integers: BigQuery cannot send int64
    parts = [patstat.df(make_query(ids[s:s + CHUNK])) for s in range(0, len(ids), CHUNK)]
    parts = [p for p in parts if len(p)]
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()


def cached(name, fetch, ids):
    "Answer of fetch(ids), saved in CACHE_DIR: a second run reads the file instead of PATSTAT."
    path = CACHE_DIR / f"{name}_{ENV}_{len(ids)}.pkl"           # the number of IDs: a new sample, a new file
    if path.exists():
        return pd.read_pickle(path)
    t0 = time.time()
    out = fetch(ids)
    out.to_pickle(path)
    print(f"{name}: {len(out):,} rows from PATSTAT in {time.time() - t0:.0f} s")
    return out


def fetch_applications(ids):
    "One row per application: office, kind (W = PCT), filing date and earliest filing date of the family."
    return in_chunks(ids, lambda part: db.query(
        TLS201_APPLN.docdb_family_id, TLS201_APPLN.appln_id, TLS201_APPLN.appln_auth,
        TLS201_APPLN.appln_kind, TLS201_APPLN.appln_filing_date, TLS201_APPLN.earliest_filing_date)
        .filter(TLS201_APPLN.docdb_family_id.in_(part), *REAL))


def fetch_cpc_subclasses(ids):
    "One row per family and CPC subclass, with the number of applications that carry it."
    return in_chunks(ids, lambda part: db.query(
        TLS201_APPLN.docdb_family_id, SUBCLASS.label("subclass"),
        func.count(distinct(TLS201_APPLN.appln_id)).label("n"))
        .join(TLS224_APPLN_CPC, TLS224_APPLN_CPC.appln_id == TLS201_APPLN.appln_id)
        .filter(TLS201_APPLN.docdb_family_id.in_(part), *REAL)
        .group_by(TLS201_APPLN.docdb_family_id, BY_SUBCLASS))


def fetch_ipc_subclasses(ids):
    "The same with IPC codes (TLS209): used only for the families without any CPC code."
    return in_chunks(ids, lambda part: db.query(
        TLS201_APPLN.docdb_family_id, IPC_SUBCLASS.label("subclass"),
        func.count(distinct(TLS201_APPLN.appln_id)).label("n"))
        .join(TLS209_APPLN_IPC, TLS209_APPLN_IPC.appln_id == TLS201_APPLN.appln_id)
        .filter(TLS201_APPLN.docdb_family_id.in_(part), *REAL)
        .group_by(TLS201_APPLN.docdb_family_id, BY_SUBCLASS))


def fetch_inventors(ids):
    "Number of distinct inventors of each family (TLS207 rows with invt_seq_nr > 0)."
    return in_chunks(ids, lambda part: db.query(
        TLS201_APPLN.docdb_family_id,
        func.count(distinct(TLS207_PERS_APPLN.person_id)).label("inventors"))
        .join(TLS207_PERS_APPLN, TLS207_PERS_APPLN.appln_id == TLS201_APPLN.appln_id)
        .filter(TLS201_APPLN.docdb_family_id.in_(part), TLS207_PERS_APPLN.invt_seq_nr > 0, *REAL)
        .group_by(TLS201_APPLN.docdb_family_id))


def fetch_forward(ids):
    "Families citing each family: one row per (cited, citing) with the citing family's first priority year."
    citing = TLS201_APPLN
    return in_chunks(ids, lambda part: db.query(
        TLS228_DOCDB_FAM_CITN.cited_docdb_family_id.label("docdb_family_id"),
        TLS228_DOCDB_FAM_CITN.docdb_family_id.label("citing_family"),
        func.min(citing.earliest_filing_year).label("citing_year"))
        .join(citing, citing.docdb_family_id == TLS228_DOCDB_FAM_CITN.docdb_family_id)
        .filter(TLS228_DOCDB_FAM_CITN.cited_docdb_family_id.in_(part), *REAL,
                TLS228_DOCDB_FAM_CITN.docdb_family_id != TLS228_DOCDB_FAM_CITN.cited_docdb_family_id)
        .group_by(TLS228_DOCDB_FAM_CITN.cited_docdb_family_id, TLS228_DOCDB_FAM_CITN.docdb_family_id))


def fetch_backward_years(ids):
    "Families cited by each family, with the cited family's first priority year (for cycle time)."
    cited = aliased(TLS201_APPLN)
    return in_chunks(ids, lambda part: db.query(
        TLS228_DOCDB_FAM_CITN.docdb_family_id,
        TLS228_DOCDB_FAM_CITN.cited_docdb_family_id.label("cited_family"),
        func.min(cited.earliest_filing_year).label("cited_year"))
        .join(cited, cited.docdb_family_id == TLS228_DOCDB_FAM_CITN.cited_docdb_family_id)
        .filter(TLS228_DOCDB_FAM_CITN.docdb_family_id.in_(part), cited.appln_id < DUMMY_ID_FROM,
                TLS228_DOCDB_FAM_CITN.docdb_family_id != TLS228_DOCDB_FAM_CITN.cited_docdb_family_id)
        .group_by(TLS228_DOCDB_FAM_CITN.docdb_family_id, TLS228_DOCDB_FAM_CITN.cited_docdb_family_id))


def fetch_backward_subclasses(ids):
    "For each family, how many of the families it cites carry each CPC subclass (for originality)."
    cited = aliased(TLS201_APPLN)
    return in_chunks(ids, lambda part: db.query(
        TLS228_DOCDB_FAM_CITN.docdb_family_id, SUBCLASS.label("subclass"),
        func.count(distinct(cited.docdb_family_id)).label("n"))
        .join(cited, cited.docdb_family_id == TLS228_DOCDB_FAM_CITN.cited_docdb_family_id)
        .join(TLS224_APPLN_CPC, TLS224_APPLN_CPC.appln_id == cited.appln_id)
        .filter(TLS228_DOCDB_FAM_CITN.docdb_family_id.in_(part), cited.appln_id < DUMMY_ID_FROM,
                TLS228_DOCDB_FAM_CITN.docdb_family_id != TLS228_DOCDB_FAM_CITN.cited_docdb_family_id)
        .group_by(TLS228_DOCDB_FAM_CITN.docdb_family_id, BY_SUBCLASS))


def fetch_npl(ids):
    "Number of distinct non-patent documents (mostly science) cited by the publications of each family."
    return in_chunks(ids, lambda part: db.query(
        TLS201_APPLN.docdb_family_id,
        func.count(distinct(TLS212_CITATION.cited_npl_publn_id)).label("npl"))
        .join(TLS211_PAT_PUBLN, TLS211_PAT_PUBLN.appln_id == TLS201_APPLN.appln_id)
        .join(TLS212_CITATION, TLS212_CITATION.pat_publn_id == TLS211_PAT_PUBLN.pat_publn_id)
        .filter(TLS201_APPLN.docdb_family_id.in_(part), *REAL,
                TLS212_CITATION.cited_npl_publn_id != "0")
        .group_by(TLS201_APPLN.docdb_family_id))


def fetch_search_letters(ids):
    "Letters (X, Y, I, A...) of the search reports of the family's own publications, all offices."
    return in_chunks(ids, lambda part: db.query(
        TLS201_APPLN.docdb_family_id, TLS215_CITN_CATEG.citn_categ,
        func.count().label("n"))
        .join(TLS211_PAT_PUBLN, TLS211_PAT_PUBLN.appln_id == TLS201_APPLN.appln_id)
        .join(TLS215_CITN_CATEG, TLS215_CITN_CATEG.pat_publn_id == TLS211_PAT_PUBLN.pat_publn_id)
        .filter(TLS201_APPLN.docdb_family_id.in_(part), *REAL)
        .group_by(TLS201_APPLN.docdb_family_id, TLS215_CITN_CATEG.citn_categ))

## 2. Basic facts of every family

The applications of every family of the net, and its **main subclass** (the subclass carried by most of its applications; ties in alphabetical order). The main subclass defines the cohort of the outcome (Section 3) and the subfield of the momentum signal (Section 4.4).

**CPC first, IPC when CPC is missing.** The development run on a sample of 20,000 families found a CPC code for only 79% of the families of the net, and for about 55% of those of 2005–2015, so scope and the cohorts were missing for many older families. Every patent has IPC codes, and at the level of the subclass (4 characters, e.g. B25J) IPC and CPC are the same scheme, so a family without CPC takes its IPC subclasses. Families with neither get the cohort of their year only.

In [ ]:
apps = cached("applications", fetch_applications, FAMILIES)
cpc = cached("cpc_subclasses", fetch_cpc_subclasses, FAMILIES)
with_cpc = set(cpc["docdb_family_id"])
no_cpc = [f for f in FAMILIES if f not in with_cpc]
ipc = cached("ipc_subclasses", fetch_ipc_subclasses, no_cpc) if no_cpc else cpc.iloc[:0]
classes = pd.concat([cpc.assign(scheme="CPC"), ipc.assign(scheme="IPC")], ignore_index=True)

main_subclass = (classes.sort_values(["docdb_family_id", "n", "subclass"], ascending=[True, False, True])
                 .drop_duplicates("docdb_family_id").set_index("docdb_family_id")["subclass"])
net["main_subclass"] = net.index.map(main_subclass)

print(f"applications: {len(apps):,} ({len(apps) / len(net):.2f} per family)")
scheme = classes.drop_duplicates("docdb_family_id").set_index("docdb_family_id")["scheme"]
print("families by source of the subclass:",
      net.index.map(scheme).fillna("none").value_counts(normalize=True).round(3).to_dict())
print("largest subclasses of the net:", net["main_subclass"].value_counts().head(8).to_dict())
check("every family has at least one application", net.index.isin(apps["docdb_family_id"]).all())

## 3. The outcome: which families became valuable?

For each family we count the **families that cite it** (TLS228) with a first priority year no later than its own plus 5, a family citing itself excluded: the same count as the Influence of Step 3. A family is **valuable** if its count is in the top 10% of its **cohort** (same priority year and main CPC subclass; the year only when the cohort has fewer than 30 families), and above zero.

Why the top and not the mean: Fleming (2001) and Lecture 6 show that value is very skewed; the question that matters is whether a patent lands in the tail. Why the cohort: in 2013 a B25J family received on average 11.8 citations within five years, a B62D family 5.8 and an A63H family 3.8 (Step 3, Section 4); comparing within the cohort makes them comparable.

The outcome is **complete only up to priority year 2017** (2017 + 5 = 2022, the last complete year). The later families are never used to train or test, only to be scored.

`TODO(Leo/Thomas)`: alternative outcomes for the robustness section (Lai et al. style: alive and highly cited against withdrawn or lapsed, TLS231; the OECD-method quality index of Step 3, Section 9.3; offices added later).

In [ ]:
fwd = cached("forward_citations", fetch_forward, FAMILIES)
fwd = fwd.merge(net[["priority_year"]], left_on="docdb_family_id", right_index=True)
in_window = fwd["citing_year"] <= fwd["priority_year"] + HORIZON
net["fwd_cites_5y"] = fwd[in_window].groupby("docdb_family_id")["citing_family"].nunique()
net["fwd_cites_5y"] = net["fwd_cites_5y"].fillna(0)


def label_top(frame, share=TOP_SHARE, min_cohort=MIN_COHORT):
    "1 if the family is in the top share of its cohort (year x subclass, else year) and cited at least once."
    cohort = frame["priority_year"].astype(str) + "|" + frame["main_subclass"].fillna("none")
    size = cohort.map(cohort.value_counts())
    cohort = cohort.where(size >= min_cohort, frame["priority_year"].astype(str))
    threshold = frame.groupby(cohort)["fwd_cites_5y"].transform(lambda s: s.quantile(1 - share))
    return ((frame["fwd_cites_5y"] >= threshold) & (frame["fwd_cites_5y"] > 0)).astype(int), cohort


labelled = net["priority_year"].between(YEAR_MIN, LAST_LABEL_YEAR)
net.loc[labelled, "valuable"], net.loc[labelled, "cohort"] = label_top(net[labelled])

by_year = net[labelled].groupby("priority_year").agg(
    families=("valuable", "size"), valuable=("valuable", "sum"),
    sector=("in_sector_v3", "sum"),
    sector_valuable=("valuable", lambda v: v[net.loc[v.index, "in_sector_v3"]].sum()))
by_year["valuable (%)"] = (100 * by_year["valuable"] / by_year["families"]).round(1)
display(by_year)
check("about 10% of the labelled families are valuable (ties can add a little)",
      0.05 <= net.loc[labelled, "valuable"].mean() <= 0.15)

## 4. The signals

Every signal is a column of the table `signals`, indexed by family. Each one is registered in `SIGNALS` with:
- **set**: `classic` (the benchmark of Lai et al.: indicators any study would use) or `ours` (what we add);
- **type**: `static` (known at publication) or `dynamic` (grows with age; at each prediction moment it may only count what had happened by then);
- **direction**: `+1` if a higher value should mean more future value, `-1` if lower;
- **paper**: where the idea comes from.

A signal must never use information from after the family's publication: this is what makes the backtest honest. Families with no value for a signal keep `NaN` (never 0): "no search report yet" is not "no X letter".

In [ ]:
signals = pd.DataFrame(index=net.index)
SIGNALS = {}


def register(name, values, set_, type_, direction, paper):
    "Add one signal to the table and describe it."
    signals[name] = values.reindex(signals.index)
    SIGNALS[name] = {"set": set_, "type": type_, "direction": direction, "paper": paper,
                     "coverage (%)": round(100 * signals[name].notna().mean(), 1)}

### 4.1 Classic indicators (the benchmark)

Scope (number of CPC subclasses), number of inventors, early international filing (PCT application; offices reached within 12 months of the first filing), backward citations and citations to non-patent literature. These are the indicators of the patent-value literature that Lai et al. (2026) use as their benchmark, restricted to what PATSTAT gives for every office and what is known at publication. Family size counted later is **not** here: it grows with age, so it would leak the future.

In [ ]:
first = apps.groupby("docdb_family_id")["earliest_filing_date"].min()
apps["days_from_first"] = (pd.to_datetime(apps["appln_filing_date"], errors="coerce")
                           - pd.to_datetime(apps["docdb_family_id"].map(first), errors="coerce")).dt.days
early = apps[(apps["days_from_first"] <= 365) & (apps["appln_auth"] != "WO")]

register("scope", classes.groupby("docdb_family_id")["subclass"].nunique(),
         "classic", "static", +1, "Lerner (1994); Lai et al. (2026)")
register("inventors", cached("inventors", fetch_inventors, FAMILIES).set_index("docdb_family_id")["inventors"],
         "classic", "static", +1, "Lai et al. (2026)")
register("pct", apps.assign(w=apps["appln_kind"].astype(str).str.strip().eq("W")
                            | apps["appln_auth"].eq("WO")).groupby("docdb_family_id")["w"].max().astype(int),
         "classic", "static", +1, "Harhoff et al. (2003); Step 3, Section 10")
register("offices_12m", early.groupby("docdb_family_id")["appln_auth"].nunique(),
         "classic", "static", +1, "Putnam (1996); Harhoff et al. (2003)")

back = cached("backward_years", fetch_backward_years, FAMILIES)
register("backward_cites", back.groupby("docdb_family_id")["cited_family"].nunique().reindex(net.index).fillna(0),
         "classic", "static", +1, "Harhoff et al. (2003); Lai et al. (2026)")
register("npl_cites", cached("npl", fetch_npl, FAMILIES).set_index("docdb_family_id")["npl"]
         .reindex(net.index).fillna(0), "classic", "static", +1, "Narin et al. (1997); Lai et al. (2026)")

### 4.2 Originality and technology cycle time

**Originality** (Trajtenberg, Henderson and Jaffe 1997; second most important indicator in Lai et al.): 1 − Σ s², where s is the share of the cited families in each CPC subclass. A family that draws on many fields scores high. **Technology cycle time** (Lai et al.): the median age of the cited families when the family was filed; a short cycle means the family builds on recent technology. Both use backward citations, which are known when the search report is published.

In [ ]:
bsub = cached("backward_subclasses", fetch_backward_subclasses, FAMILIES)
shares = bsub["n"] / bsub.groupby("docdb_family_id")["n"].transform("sum")
originality = 1 - (shares ** 2).groupby(bsub["docdb_family_id"]).sum()
register("originality", originality, "ours", "static", +1, "Trajtenberg, Henderson and Jaffe (1997)")

age = back.merge(net[["priority_year"]], left_on="docdb_family_id", right_index=True)
age["age"] = age["priority_year"] - age["cited_year"]
register("cycle_time", age[age["age"] >= 0].groupby("docdb_family_id")["age"].median(),
         "ours", "static", -1, "Lai et al. (2026)")

### 4.3 The examiner's opinion

Did the examiner find a document that threatens the novelty of the family (**X**, or **I** at the EPO) or its inventive step together with another one (**Y**)? The letters come from the search reports of **all offices** in PATSTAT (TLS215), which cover 58% of the sector, against 11 families with EPAB (`step4_search_reports_check.ipynb`). Czarnitzki, Hussinger and Leten (2011) show that the letters carry information on value; Lemley and Sampat (2012) that examiners differ, so the office of the report is a control (`TODO(Leo/Thomas)`: add it).

The signal is **no X/Y/I** (1 = the examiner found nothing close): a family without any report stays empty. It is static once the report exists, but the report can come one or two years after filing, so for the most recent families it is often missing.

In [ ]:
letters = cached("search_letters", fetch_search_letters, FAMILIES)
letters["close"] = letters["citn_categ"].astype(str).str.strip().str.upper().isin(["X", "Y", "I"])
close_any = letters.groupby("docdb_family_id")["close"].max()
register("no_close_prior_art", (~close_any).astype(int), "ours", "static", +1,
         "Czarnitzki, Hussinger and Leten (2011); Lemley and Sampat (2012)")

### 4.4 Early entry into a hot subfield

The family's main CPC subclass grew faster than all of robotics in the three years before the family was filed (momentum known at filing): families in the net with the same subclass in years y−3 to y−1, against years y−6 to y−4, divided by the same ratio for the whole net. Values above 1 mean the subfield was heating up. This is the course's construct of the tutorial of 9 November; Steps 0 and 2 showed that control, legs and gait planning grew while feet shrank.

`TODO(Leo/Thomas)`: a version on the subfields platform / body / control for the sector families (Step 0, 7.4), and "early in its curve" (the family is among the first of the subfield).

In [ ]:
counts = net.groupby(["main_subclass", "priority_year"]).size()
total = net.groupby("priority_year").size()


def window_sum(series, years):
    return series.reindex(years, fill_value=0).sum()


momentum = {}
for (sub, year), _ in net.groupby(["main_subclass", "priority_year"]).size().items():
    recent, before = range(year - 3, year), range(year - 6, year - 3)
    s_recent, s_before = window_sum(counts[sub], recent), window_sum(counts[sub], before)
    t_ratio = window_sum(total, recent) / max(window_sum(total, before), 1)
    momentum[(sub, year)] = (s_recent / s_before) / t_ratio if s_before >= 5 else np.nan
key = list(zip(net["main_subclass"], net["priority_year"]))
register("subfield_momentum", pd.Series([momentum.get(k, np.nan) for k in key], index=net.index),
         "ours", "static", +1, "course construct (tutorial of 9 November)")

### 4.5 Text novelty (sector only for now)

1 − mean similarity of the abstract with the 10 closest abstracts of the previous 5 years (Patent SBERT), from notebook 21, Section 10 (Kelly et al. 2021). It exists only for the 2,765 sector families with an abstract, so it cannot enter a model trained on the whole net: it is tested as a **baseline on the sector** (Section 5.3). Notebook 21 showed that novelty falls as the sector grows, so we use its **rank within the year**.

`TODO(Leo)`: the same signal on the wide net (encode the English abstracts of the net with Patent SBERT; about 240,000 abstracts, to run once and save), and the second version of Arts, Hou and Gomez (2021): new words and word pairs never seen before.

In [ ]:
SECTOR_ONLY = []
if text is not None:
    register("text_novelty_rank", text.set_index("docdb_family_id")["novelty_rank_in_year"],
             "ours", "static", +1, "Kelly et al. (2021); notebook 21")
    SECTOR_ONLY.append("text_novelty_rank")

### 4.6 Signals still to build

- `TODO(Thomas)` **New recombination** (Fleming 2001; Verhoeven, Bakker and Veugelers 2016): the share of the family's pairs of CPC groups never seen together in PATSTAT before its priority year. It needs the first year of every pair of groups in PATSTAT: a self-join of TLS224 restricted to the groups of the net, run once inside PATSTAT and saved.
- `TODO(Leo)` **Disruption** (Funk and Owen-Smith 2017, CD index) and **SPNP** (Lai et al. 2026) on the citation network. These are **dynamic**: at each prediction moment they may only use the citations received by then (Section 5.4).
- `TODO(Thomas)` **Firm-level signals**: growth of the firm's filings in the sector, variety of its subfields, share of its families with PCT.
- Links to science on US patents (Reliance on Science, Step 3, 3.5) cover only 171 families of the sector: a check, not a model input.

Each new signal: one function, one `register(...)` call with its paper, then rerun Sections 5 to 7.

In [ ]:
signal_table = pd.DataFrame(SIGNALS).T
display(signal_table)

# coverage by priority year: a signal empty in the recent years cannot score the firms of 2023-2025
coverage = signals.notna().groupby(net["priority_year"]).mean().mul(100).round(0)
display(coverage.loc[[y for y in (2005, 2010, 2015, 2017, 2020, 2023, 2024, 2025) if y in coverage.index]])

## 5. The backtest: does the score work?

**Out of time.** To predict the families of a test year T, the model may only learn from families whose 5-year outcome was known in T, that is, priority years T−5−LOOKBACK+1 to T−5. For T = 2013 it learns on 2004–2008; for T = 2017 on 2008–2012. Lai et al. use random folds, which mix past and future; a score about the future must be tested on the future (Aristodemou).

**Benchmark against proposed** (Lai et al.): the same models with the classic indicators only, then with our signals added. **Two models**: a logistic regression (simple, its signs can be read) and gradient boosting (it handles missing values and interactions; Lai et al. found XGBoost best).

**Metrics** on the real, unbalanced distribution: AUC, and the **precision in the top 10%** (of the families the model ranks in its top 10%, how many were really valuable; chance = about 10%). Each with a bootstrap 95% interval. Results for all test families of the net, and for the **sector's families only** (pooled over the test years, because each year has few).

In [ ]:
CLASSIC = [s for s, d in SIGNALS.items() if d["set"] == "classic"]
OURS = [s for s, d in SIGNALS.items() if d["set"] == "ours" and s not in SECTOR_ONLY]
FEATURE_SETS = {"benchmark": CLASSIC, "proposed": CLASSIC + OURS}


def make_models():
    "The two models; both treat missing values explicitly."
    logistic = Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True)),
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced")),
    ])
    boosting = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,
                                              class_weight="balanced", random_state=RANDOM_STATE)
    return {"logistic": logistic, "boosting": boosting}


def precision_at_top(y, score, share=TOP_SHARE):
    "Share of valuable families among the share of families with the highest scores."
    k = max(1, int(round(share * len(y))))
    top = np.argsort(-np.asarray(score))[:k]
    return float(np.asarray(y)[top].mean())


def with_interval(metric, y, score, n_boot=N_BOOT):
    "The metric and its bootstrap 95% interval (resampling families)."
    y, score = np.asarray(y), np.asarray(score)
    rng = np.random.default_rng(RANDOM_STATE)
    value = metric(y, score)
    draws = []
    for _ in range(n_boot):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() != y[i].max():                 # a resample with one class has no AUC
            draws.append(metric(y[i], score[i]))
    low, high = np.percentile(draws, [2.5, 97.5]) if draws else (np.nan, np.nan)
    return value, low, high


def split_for(test_year):
    "Training and test families for one test year, out of time."
    train_years = range(test_year - HORIZON - LOOKBACK + 1, test_year - HORIZON + 1)
    known = net["valuable"].notna()
    train = net.index[known & net["priority_year"].isin(train_years)]
    test = net.index[known & (net["priority_year"] == test_year)]
    return train, test

In [ ]:
predictions = []                                    # one row per test family, model and feature set
for T in TEST_YEARS:
    train, test = split_for(T)
    y_train = net.loc[train, "valuable"].astype(int)
    for set_name, features in FEATURE_SETS.items():
        for model_name, model in make_models().items():
            model.fit(signals.loc[train, features], y_train)
            score = model.predict_proba(signals.loc[test, features])[:, 1]
            predictions.append(pd.DataFrame({
                "docdb_family_id": test, "test_year": T, "set": set_name, "model": model_name,
                "score": score, "valuable": net.loc[test, "valuable"].astype(int).values,
                "in_sector": net.loc[test, "in_sector_v3"].values}))
    print(f"T = {T}: trained on {len(train):,} families ({int(y_train.sum()):,} valuable), "
          f"tested on {len(test):,}")
predictions = pd.concat(predictions, ignore_index=True)
predictions.to_csv(OUT_DIR / "backtest_predictions.csv", index=False)

In [ ]:
def summarise(frame):
    "AUC and precision in the top 10%, with intervals, for one group of predictions."
    auc = with_interval(roc_auc_score, frame["valuable"], frame["score"])
    p10 = with_interval(precision_at_top, frame["valuable"], frame["score"])
    return pd.Series({"families": len(frame), "valuable": int(frame["valuable"].sum()),
                      "AUC": round(auc[0], 3), "AUC 95%": f"{auc[1]:.3f}-{auc[2]:.3f}",
                      "precision top 10%": round(p10[0], 3), "precision 95%": f"{p10[1]:.3f}-{p10[2]:.3f}"})


results_net = predictions.groupby(["model", "set"]).apply(summarise, include_groups=False)
results_sector = (predictions[predictions["in_sector"]].groupby(["model", "set"])
                  .apply(summarise, include_groups=False))
auc_by_year = predictions.groupby(["test_year", "model", "set"]).apply(
    lambda f: round(roc_auc_score(f["valuable"], f["score"]), 3), include_groups=False).unstack("set")
print("all test families of the net, pooled over the test years:")
display(results_net)
print("the sector's test families only:")
display(results_sector)
print("AUC by test year:")
display(auc_by_year)

### 5.3 Baselines: each signal alone

The professor asks to beat the baselines of the literature. Each signal alone ranks the test families (in its direction), with no model: if the proposed model is not better than the best single signal, it adds nothing. Text novelty is tested here on the sector's families only.

In [ ]:
test_ids = predictions.loc[(predictions["model"] == "logistic") & (predictions["set"] == "benchmark"),
                           ["docdb_family_id", "valuable", "in_sector"]]
rows = []
for name, d in SIGNALS.items():
    for population, ids in (("net", test_ids), ("sector", test_ids[test_ids["in_sector"]])):
        if population == "net" and name in SECTOR_ONLY:
            continue                                 # a sector-only signal is tested on the sector only
        values = signals.loc[ids["docdb_family_id"], name].values * d["direction"]
        ok = ~np.isnan(values)
        if ok.sum() < 30 or ids["valuable"].values[ok].min() == ids["valuable"].values[ok].max():
            continue                                 # too few families, or only one class
        auc, low, high = with_interval(roc_auc_score, ids["valuable"].values[ok], values[ok])
        rows.append({"signal": name, "population": population, "families": int(ok.sum()),
                     "AUC": round(auc, 3), "AUC 95%": f"{low:.3f}-{high:.3f}"})
baselines = pd.DataFrame(rows).sort_values(["population", "AUC"], ascending=[True, False])
display(baselines)

### 5.4 Prediction moments (Lai et al.)

`TODO(Leo/Thomas)`: repeat the backtest at publication (now), and 2, 3 and 5 years after it, adding the **dynamic** signals counted only up to that moment (citations received so far, CD index, SPNP, offices added so far). Report how much is lost by predicting early: this is Lai et al.'s third question, and the answer tells Electrolux how much to trust a score of a 2025 patent.

## 6. Which signals matter, and in which direction

Two readings, as in Lai et al. (their tables 11 and 12): the **permutation importance** of each signal in gradient boosting (how much the AUC drops when the signal is shuffled), and the **signs** of the logistic regression on standardised signals. Both on the last test year.

In [ ]:
T = TEST_YEARS[-1]
train, test = split_for(T)
features = FEATURE_SETS["proposed"]
models = make_models()
for model in models.values():
    model.fit(signals.loc[train, features], net.loc[train, "valuable"].astype(int))

imp = permutation_importance(models["boosting"], signals.loc[test, features],
                             net.loc[test, "valuable"].astype(int), scoring="roc_auc",
                             n_repeats=5, random_state=RANDOM_STATE)
coef = models["logistic"].named_steps["model"].coef_[0][:len(features)]
importance = pd.DataFrame({"AUC drop when shuffled": imp.importances_mean.round(4),
                           "logistic coefficient (standardised)": coef.round(3),
                           "expected direction": [SIGNALS[f]["direction"] for f in features]},
                          index=features).sort_values("AUC drop when shuffled", ascending=False)
display(importance)

## 7. The future score of the firms

**Which model.** Rule: the model and the set of signals with the highest AUC on all the test families of the net (Section 5). We fixed this rule after the development run on a sample of 20,000 families, where gradient boosting was below the logistic regression, and before the full run; the first version of the notebook used gradient boosting by default.

The final model learns on the last five years with a known outcome (2013–2017) and scores the **sector's families of 2023–2025**. A firm's future score is the **mean predicted probability** of its recent families (each family counted once for each of its applicants), shown with the number of families behind it; firms with fewer than 5 recent families get no score.

**Which firms** (open decision, see the doc): for now the firms with a present score (Step 3) and the firms waiting for Step 4. `TODO(group)`: decide whether to use Thomas's panel of 5 entrants, 5 medium and 5 large firms counted up to 2025, with the rule fixed before looking at the scores.

**A warning to keep in the deck:** the model learns on families of 2013–2017 and scores families of 2023–2025. If the sector changed in between (it did: it became Chinese and grew fast), the scores are less reliable than the backtest says. Section 5.4 measures part of this.

In [ ]:
# 1. final model: the best model and set of Section 5, trained on the last LOOKBACK labelled years
best_model, best_set = results_net["AUC"].astype(float).idxmax()
features = FEATURE_SETS[best_set]
print(f"final model: {best_model} with the {best_set} signals (AUC on the net {results_net.loc[(best_model, best_set), 'AUC']})")
final_years = range(LAST_LABEL_YEAR - LOOKBACK + 1, LAST_LABEL_YEAR + 1)
train = net.index[net["valuable"].notna() & net["priority_year"].isin(final_years)]
final = make_models()[best_model].fit(signals.loc[train, features], net.loc[train, "valuable"].astype(int))

# 2. the sector's recent families, scored
recent = net.index[net["in_sector_v3"] & net["priority_year"].between(*APPLY_YEARS)]
family_score = pd.Series(final.predict_proba(signals.loc[recent, features])[:, 1],
                         index=recent, name="probability")

# 3. the firm of each family: the brand key of Step 0, with the merges of Step 3, Section 2.2
MERGE = {"UNITREE": ["HANGZHOU YUSHU TECH"], "AGIBOT": ["SHANGHAI ZHIYUAN XINCHUANG TECHNOLOGY"],
         "XIAOMI ROBOT": ["BEIJING XIAOMI ROBOT TECH"],
         "HONDA RESEARCH INSTITUTE EUROPE": ["HONDA RESEARCH INST EUROPE", "HONDA RESEARCH INSTITUTE EUROPE"],
         "ALDEBARAN ROBOTICS": ["ALDEBARAN ROBOTICS", "软银机器人欧洲公司", "奥尔德巴伦机器人公司"],
         "PANASONIC": ["PANASONIC", "MATSUSHITA ELECT IND"]}
to_firm = {name: firm for firm, names in MERGE.items() for name in names}
fam_firm = family_applicants[["docdb_family_id", "applicant"]].copy()
fam_firm["firm"] = fam_firm["applicant"].map(to_firm).fillna(fam_firm["applicant"])
fam_firm = fam_firm.drop_duplicates(["docdb_family_id", "firm"])

FIRMS = sorted(set(present["firm"]) | set(waiting["firm"]))      # TODO(group): the final list of firms
scored = fam_firm[fam_firm["firm"].isin(FIRMS)].merge(family_score, left_on="docdb_family_id",
                                                      right_index=True)
future = scored.groupby("firm").agg(recent_families=("probability", "size"),
                                    mean_probability=("probability", "mean"))
future.loc[future["recent_families"] < MIN_RECENT_FAMILIES, "mean_probability"] = np.nan
future["expected_valuable_families"] = scored.groupby("firm")["probability"].sum().round(1)
future["rank"] = future["mean_probability"].rank(ascending=False, method="min")
future = future.reindex(FIRMS).sort_values("rank")
display(future.round(3))
future.reset_index(names="firm").to_csv(OUT_DIR / "future_score.csv", index=False)   # read by Step 5
print(f"saved {OUT_DIR / 'future_score.csv'}")

## 8. Checks before using the results

In [ ]:
check("no family of the scored years was used to train or test",
      not set(recent) & set(predictions["docdb_family_id"]))
check("the training years of every test year end 5 years before it",
      all(net.loc[split_for(T)[0], "priority_year"].max() <= T - HORIZON for T in TEST_YEARS))
check("every signal is registered with its paper and type",
      all(d["paper"] and d["type"] in ("static", "dynamic") for d in SIGNALS.values()))
check("no dynamic signal in the models yet (they need Section 5.4)",
      all(SIGNALS[s]["type"] == "static" for s in FEATURE_SETS["proposed"]))
check("every firm of the list is in the output", set(future.index) == set(FIRMS))

## 9. Answer for the deck

`TODO(group)`, after the run, in the same form as Steps 1–3:
1. Does the proposed score beat the benchmark and the single signals, on the net and on the sector? By how much, with intervals?
2. Which signals matter most, and do their signs make sense?
3. Which firms have the highest future score, and how many recent families is each score based on?
4. Limits: the outcome is citations (an imperfect proxy of value); the examiner letters are mostly Chinese; the model learns on 2013–2017 and scores 2023–2025.